## Watch a Smart Agent!

### 1.Start the Environment for Trained Agent

In [6]:
import numpy as np
import torch
import gymnasium as gym
import os
import time
import matplotlib.pyplot as plt
from IPython import display as ipython_display

from agent import Agent

# MPS lacks aten::_sample_dirichlet needed by Beta distribution sampling.
# CPU is fast enough for inference and avoids the error.
device = torch.device("cpu")
print(device)
torch.set_default_dtype(torch.float32)

def rgb2gray(rgb, norm=True):
        # rgb image -> gray [0, 1]
    gray = np.dot(rgb[..., :], [0.299, 0.587, 0.114])
    if norm:
        # normalize
        gray = gray / 128. - 1.
    return gray

seed = 0
img_stack = 4
action_repeat = 10
env = gym.make('CarRacing-v3', render_mode='rgb_array', verbose=0)
state = env.reset()
reward_threshold = env.spec.reward_threshold

cpu


In [12]:
class Wrapper():
    """
    Environment wrapper for CarRacing 
    """

    def __init__(self, env):
        self.env = env  

    def reset(self):
        self.counter = 0
        self.av_r = self.reward_memory()

        self.die = False
        img_rgb, _ = env.reset()
        img_gray = rgb2gray(img_rgb)
        self.stack = [img_gray] * img_stack  # four frames for decision
        return np.array(self.stack)

    def step(self, action):
        total_reward = 0
        for i in range(action_repeat):
            img_rgb, reward, die, _, _ = env.step(action)
            # don't penalize "die state"
            if die:
                reward += 100
            # green penalty
            if np.mean(img_rgb[:, :, 1]) > 185.0:
                reward -= 0.05
            total_reward += reward
            # if no reward recently, end the episode
            done = True if self.av_r(reward) <= -0.1 else False
            if done or die:
                break
        img_gray = rgb2gray(img_rgb)
        self.stack.pop(0)
        self.stack.append(img_gray)
        assert len(self.stack) == img_stack
        return np.array(self.stack), total_reward, done, die


    @staticmethod
    def reward_memory():
        # record reward for last 100 steps
        count = 0
        length = 100
        history = np.zeros(length)

        def memory(reward):
            nonlocal count
            history[count] = reward
            count = (count + 1) % length
            return np.mean(history)

        return memory

    
agent = Agent(device)

env_wrap = Wrapper(env)    

### 2. Prepare Load

In [8]:
def load(agent, directory, filename):
    state_dict = torch.load(
        os.path.join(directory, filename),
        map_location='cpu',    # load to CPU first
        weights_only=True
    )
    # convert any float64 weights → float32
    state_dict = {k: v.float() for k, v in state_dict.items()}
    
    agent.net.load_state_dict(state_dict)
    agent.net.float().to(device)   # ensure float32 on MPS
    print(f"Loaded {filename} → {device}")


### 3. Prepare Player

In [9]:
from collections import deque
import os

def play(env, agent, n_episodes):
    state = env_wrap.reset()
    
    scores_deque = deque(maxlen=100)
    scores = []
    
    for i_episode in range(1, n_episodes+1):
        state = env_wrap.reset()        
        score = 0
        
        time_start = time.time()
        
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        plt.axis("off")
        img_handle = None

        while True:
            action, a_logp = agent.select_action(state)
            frame = env.render()
            if img_handle is None:
                img_handle = ax.imshow(frame)
            else:
                img_handle.set_data(frame)
            ipython_display.display(fig)
            ipython_display.clear_output(wait=True)
            next_state, reward, done, die = env_wrap.step( \
                action * np.array([2., 1., 1.]) + np.array([-1., 0., 0.]))

            state = next_state
            score += reward
            
            if done or die:
                break 

        plt.close(fig)
        s = (int)(time.time() - time_start)
        
        scores_deque.append(score)
        scores.append(score)

        print('Episode {}\tAverage Score: {:.2f},\tScore: {:.2f} \tTime: {:02}:{:02}:{:02}'\
                  .format(i_episode, np.mean(scores_deque), score, s//3600, s%3600//60, s%60))  


### 3. Load and Play: Score = 350-550

In [13]:
load(agent, 'dir_chk', 'model_weights_350-550.pth')
play(env, agent, n_episodes=5)

Episode 5	Average Score: 715.30,	Score: 607.68 	Time: 00:00:07


### 4. Load and Play: Score = 580-660

In [14]:
load(agent, 'dir_chk', 'model_weights_480-660.pth')
play(env, agent, n_episodes=5)

Episode 5	Average Score: 871.35,	Score: 939.30 	Time: 00:00:07


### 5. Load and Play: Score = 820-980

In [ ]:
load(agent, 'dir_chk', 'model_weights_820-980.pth')
play(env, agent, n_episodes=5)

In [ ]:
env.close()